# Estimating Policy Impact Using Machine Learning

## Approach: Random Forest Model

### What is Random Forest?
- A **machine learning model** that makes predictions by combining many decision trees.
- This method relies on these decision trees to provide a final prediction.
- It **averages their answers**, leading to more stable and reliable predictions.

### Why Random Forest?
✅ **Works Well with Text Data**: Since we only have policy descriptions and titles (not structured scores), this model helps find patterns without needing manual classification.

✅ **Handles Low-Quality Data**: Our database (*iea_policies_raw*) has inconsistent formats, making manual research difficult. Using text processing (TF-IDF), Random Forest helps **extract useful insights without reading every record**.

✅ **Reduces Overfitting**: Overfitting happens when a model memorizes the training data instead of learning patterns. Random Forest avoids this by combining multiple trees, preventing it from relying too much on any single pattern.

### Process
1. **Convert Text into Numbers**: TF-IDF vectorization of policy descriptions/titles.
2. **Train the Model**: Random Forest learns from policies with known impact values.
3. **Predict Missing Values**: It estimates policy impact for records without predefined weights.

### Key Benefit
This method automates impact assessment, making it a **practical alternative to manually reading thousands of policies** while still capturing meaningful differences in restrictiveness.

This is an **initial test** to see if machine learning can help categorize policies efficiently. Further improvements can be made if this approach is pursued.

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Datasets
iea_pol_per_yr = pd.read_csv("/Users/cynthiasalinas/Documents/econ_climate/iea_pol_per_year.csv")
iea_policies_raw = pd.read_csv("/Users/cynthiasalinas/Documents/econ_climate/IEA_policies_raw.csv", encoding="latin1")


iea_pol_per_yr.head()


,country,iso3,year,pol_count
0,Morocco,MAR,2011,1
1,Morocco,MAR,2012,1
2,Morocco,MAR,2013,1
3,Morocco,MAR,2014,1
4,Morocco,MAR,2015,1


In [12]:
#Remove special char
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

iea_policies_raw["clean_description"] = iea_policies_raw["description"].apply(clean_text)

# Vectorization
vectorizer = TfidfVectorizer(max_features=100)  # Max 100 words
policy_text_matrix = vectorizer.fit_transform(iea_policies_raw["clean_description"])

# Create variable objectives
policy_weights = {
    "tax": 5,
    "emissions trading": 4,
    "carbon price": 4,
    "fuel efficiency": 3,
    "subsidy": 2,
    "energy efficiency": 1
}

# Function to assign weight to the existing policies
def assign_weight(description):
    description = str(description).lower()
    for keyword, weight in policy_weights.items():
        if keyword in description:
            return weight
    return 1  # Default if there are no coincidences

iea_policies_raw["policy_weight"] = iea_policies_raw["description"].apply(assign_weight)

# Train and test
X_train, X_test, y_train, y_test = train_test_split(policy_text_matrix, iea_policies_raw["policy_weight"], test_size=0.2, random_state=42)

# Using ML (Random Forest)
policy_impact_model = RandomForestRegressor(n_estimators=100, random_state=42)
policy_impact_model.fit(X_train, y_train)

# Predicting impact in new policies
iea_policies_raw["predicted_impact"] = policy_impact_model.predict(policy_text_matrix)

# Merge with iea_pol_per_yr add impact to each policy per year
iea_pol_with_impact = iea_pol_per_yr.merge(iea_policies_raw[["ISO3", "year", "predicted_impact"]],
                                           left_on=["iso3", "year"],
                                           right_on=["ISO3", "year"],
                                           how="left")

print(iea_pol_with_impact.head(15))

    country iso3  year  pol_count ISO3  predicted_impact
0   Morocco  MAR  2011          1  MAR          1.320000
1   Morocco  MAR  2011          1  MAR          1.340000
2   Morocco  MAR  2012          1  NaN               NaN
3   Morocco  MAR  2013          1  NaN               NaN
4   Morocco  MAR  2014          1  MAR          1.000000
5   Morocco  MAR  2015          1  MAR          1.000000
6   Morocco  MAR  2015          1  MAR          1.040000
7   Morocco  MAR  2015          1  MAR          1.040000
8   Morocco  MAR  2016          1  MAR          1.000000
9   Morocco  MAR  2016          1  MAR          1.320000
10  Morocco  MAR  2017          1  MAR          1.630000
11  Morocco  MAR  2017          1  MAR          1.010000
12  Morocco  MAR  2018          1  MAR          1.130000
13  Morocco  MAR  2018          1  MAR          1.054872
14  Morocco  MAR  2019          1  MAR          1.000000


In [8]:
# Descriptive Statistics
print(iea_pol_with_impact["predicted_impact"].describe())

# Contar valores NaN
print("\nCantidad de NaNs en predicted_impact:", iea_pol_with_impact["predicted_impact"].isna().sum())


count    779729.000000
mean          1.379590
std           0.719708
min           1.000000
25%           1.030000
50%           1.120000
75%           1.320000
max           5.000000
Name: predicted_impact, dtype: float64

Cantidad de NaNs en predicted_impact: 8033


In [11]:
#NaaNs
print("Cantidad de NaNs en predicted_impact:", iea_pol_with_impact["predicted_impact"].isna().sum())
df_nans = iea_pol_with_impact[iea_pol_with_impact["predicted_impact"].isna()]
print(df_nans.head())
print("Distribución de NaNs por país:")
print(df_nans["country"].value_counts())


Cantidad de NaNs en predicted_impact: 8033
      country iso3  year  pol_count ISO3  predicted_impact
2     Morocco  MAR  2012          1  NaN               NaN
3     Morocco  MAR  2013          1  NaN               NaN
32    Morocco  MAR  2012          2  NaN               NaN
33    Morocco  MAR  2013          2  NaN               NaN
2677  Finland  FIN  2012          1  NaN               NaN
Distribución de NaNs por país:
country
Islamic Republic of Iran    171
Belarus                     122
Jordan                      116
Tunisia                     112
Russian Federation          112
                           ... 
Sudan                         1
Holy See                      1
Monaco                        1
San Marino                    1
Somalia                       1
Name: count, Length: 189, dtype: int64
